# Bước 4 — Transformation & giảm chiều (Đồ án 5)
**Dự đoán tái nhập viện trong 30 ngày** · Chạy hoàn toàn trên máy cá nhân.

**Nguyên tắc:** Chia **theo bệnh nhân trước**; chỉ học imputer/scaler/encoder/PCA trên **train**. Không dùng `readmitted`, `readmitted_30d`, `patient_nbr`, `encounter_id` làm đặc trưng. Cần kiểm tra tính sẵn có của biến tại thời điểm **xuất viện** trước khi dùng trong mô hình.

Notebook này tạo ba tập train/validation/test, xử lý biến số và danh mục, lưu ma trận thưa và ghi nhật ký quyết định. Không huấn luyện mô hình phân lớp tại bước này.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import scipy.sparse as sp
from IPython.display import display
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
import joblib

RANDOM_STATE = 42
cwd = Path.cwd()
root = next((p for p in [cwd, *cwd.parents] if (p/'data'/'processed'/'step3_cleaned.csv').exists()), None)
if root is None:
    raise FileNotFoundError('Không tìm thấy data/processed/step3_cleaned.csv. Hãy mở JupyterLab từ D:/Readmission_Project.')
source = root/'data'/'processed'/'step3_cleaned.csv'
out = root/'data'/'processed'/'step4'
report = root/'reports'/'transformation'
out.mkdir(parents=True, exist_ok=True)
report.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(source, low_memory=False)
assert {'encounter_id','patient_nbr','readmitted','readmitted_30d'}.issubset(df.columns)
assert df['encounter_id'].is_unique, 'Encounter ID cần được kiểm tra trước khi chạy.'
assert df['readmitted_30d'].isin([0,1]).all()
print('Nguồn:',source)
print('Kích thước:',df.shape)
print('Bệnh nhân:',df.patient_nbr.nunique())

## 1. Chọn đặc trưng theo thời điểm dự đoán
Bài toán giả định dự đoán **ngay lúc xuất viện**. Loại biến định danh, nhãn, và mã tình trạng xuất viện (`discharge_disposition_id`) trong bản mô hình cơ sở để tránh dùng thông tin kết cục/chăm sóc sau xuất viện sai mục đích. Các biến được chọn phải được xác nhận có sẵn vào thời điểm dự đoán; nếu triển khai sớm hơn xuất viện, phải xét lại biến thời gian nằm viện và điều trị.

**Chú ý:** Đây là danh sách đặc trưng để thử nghiệm, không khẳng định chúng có tính nhân quả.

In [ ]:
numeric_candidates = [
    'time_in_hospital','num_lab_procedures','num_procedures','num_medications',
    'number_outpatient','number_emergency','number_inpatient','number_diagnoses',
    'A1Cresult_tested','max_glu_serum_tested'
]
category_candidates = [
    'race','gender','age','admission_type_id_category','admission_source_id_category',
    'payer_code','medical_specialty','diag_1','diag_2','diag_3',
    'max_glu_serum','A1Cresult','change','diabetesMed',
    'insulin','metformin','repaglinide','nateglinide','chlorpropamide','glimepiride',
    'acetohexamide','glipizide','glyburide','tolbutamide','pioglitazone',
    'rosiglitazone','acarbose','miglitol','troglitazone','tolazamide',
    'examide','citoglipton','glyburide-metformin','glipizide-metformin',
    'glimepiride-pioglitazone','metformin-rosiglitazone','metformin-pioglitazone'
]
num_cols=[c for c in numeric_candidates if c in df.columns]
cat_cols=[c for c in category_candidates if c in df.columns]
missing=[c for c in numeric_candidates+category_candidates if c not in df.columns]
if missing: print('Lưu ý: Một số cột ứng viên không tồn tại:',missing)
assert len(num_cols)>0 and len(cat_cols)>0
assert not set(['encounter_id','patient_nbr','readmitted','readmitted_30d','discharge_disposition_id']) & set(num_cols+cat_cols)
features=num_cols+cat_cols
print('Số biến số:',len(num_cols),'| biến phân loại:',len(cat_cols),'| tổng:',len(features))
print('Các biến đầu vào:',features)

## 2. Chia dữ liệu theo bệnh nhân (70% / 15% / 15% lượt khám, xấp xỉ)
Dùng GroupShuffleSplit hai lần: nhóm bệnh nhân không được xuất hiện ở nhiều tập. Tỉ lệ thực tế tính theo lượt khám có thể chênh nhẹ; notebook sẽ báo cáo tỷ lệ và mức cân bằng nhãn.

In [ ]:
groups=df['patient_nbr']
idx=np.arange(len(df))
split1=GroupShuffleSplit(n_splits=1,test_size=0.30,random_state=RANDOM_STATE)
train_idx,temp_idx=next(split1.split(idx,df['readmitted_30d'],groups))
split2=GroupShuffleSplit(n_splits=1,test_size=0.50,random_state=RANDOM_STATE)
val_rel,test_rel=next(split2.split(temp_idx,df.iloc[temp_idx]['readmitted_30d'],groups.iloc[temp_idx]))
val_idx=temp_idx[val_rel]
test_idx=temp_idx[test_rel]
splits={'train':train_idx,'validation':val_idx,'test':test_idx}
patient_sets={k:set(df.iloc[ix]['patient_nbr']) for k,ix in splits.items()}
assert all(patient_sets[a].isdisjoint(patient_sets[b]) for a,b in [('train','validation'),('train','test'),('validation','test')])
assert len(set(np.concatenate(list(splits.values()))))==len(df)
rows=[]
for name,ix in splits.items():
    subset=df.iloc[ix]
    rows.append({'tap':name,'so_luot':len(ix),'ty_le_luot_%':round(100*len(ix)/len(df),2),
                 'so_benh_nhan':subset.patient_nbr.nunique(),
                 'nhan_1':int(subset.readmitted_30d.sum()),
                 'ty_le_nhan_1_%':round(100*subset.readmitted_30d.mean(),3)})
summary=pd.DataFrame(rows)
display(summary)
summary.to_csv(report/'split_summary.csv',index=False,encoding='utf-8-sig')
for name,ix in splits.items():
    df.iloc[ix][['encounter_id','patient_nbr','readmitted_30d']].to_csv(out/f'{name}_ids_labels.csv',index=False)
print('Xác nhận: KHÔNG bệnh nhân nào giao nhau giữa train/validation/test.')

## 3. Biến đổi chỉ fit trên train
- Numeric: median imputation → StandardScaler.
- Category: thay thiếu bằng `Unknown` → OneHotEncoder(`handle_unknown='ignore'`) để không lỗi khi tập val/test có danh mục mới. Giới hạn tần suất hiếm (`min_frequency=50`) để giảm số cột quá lớn.
- Chỉ `fit` trên train, sau đó `transform` validation/test.

**Cảnh báo:** Không gộp train+validation+test trước khi tính median, trung bình/độ lệch chuẩn hay danh mục OneHot.

In [ ]:
X_train=df.iloc[train_idx][features].copy()
X_val=df.iloc[val_idx][features].copy()
X_test=df.iloc[test_idx][features].copy()
y_train=df.iloc[train_idx]['readmitted_30d'].astype('int8').to_numpy()
y_val=df.iloc[val_idx]['readmitted_30d'].astype('int8').to_numpy()
y_test=df.iloc[test_idx]['readmitted_30d'].astype('int8').to_numpy()
for frame in [X_train,X_val,X_test]:
    for c in num_cols: frame[c]=pd.to_numeric(frame[c],errors='coerce')
    for c in cat_cols: frame[c]=frame[c].astype('string').fillna('Unknown').replace({'?':'Unknown','':'Unknown'}).astype(str)
num_pipe=Pipeline([('imputer',SimpleImputer(strategy='median')),('scaler',StandardScaler())])
cat_pipe=Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),
                   ('onehot',OneHotEncoder(handle_unknown='ignore',min_frequency=50,sparse_output=True))])
processor=ColumnTransformer([('numeric',num_pipe,num_cols),('categorical',cat_pipe,cat_cols)],
                            sparse_threshold=1.0)
Xtr=processor.fit_transform(X_train)
Xva=processor.transform(X_val)
Xte=processor.transform(X_test)
Xtr=sp.csr_matrix(Xtr); Xva=sp.csr_matrix(Xva); Xte=sp.csr_matrix(Xte)
assert Xtr.shape[1]==Xva.shape[1]==Xte.shape[1]
assert np.isfinite(Xtr.data).all() and np.isfinite(Xva.data).all() and np.isfinite(Xte.data).all()
print('Train:',Xtr.shape,'Validation:',Xva.shape,'Test:',Xte.shape)
print('Sparsity train:',round(100*(1-Xtr.nnz/(Xtr.shape[0]*Xtr.shape[1])),2),'%')

## 4. Xuất ma trận và pipeline
Ma trận đã biến đổi lưu dạng `.npz` (sparse) thay vì CSV khổng lồ; danh sách đặc trưng và pipeline `.joblib` cho phép biến đổi dữ liệu mới bằng cùng tham số. File `test` dùng để đánh giá cuối cùng, không dùng để chỉnh mô hình.

In [ ]:
for name,matrix in [('train',Xtr),('validation',Xva),('test',Xte)]:
    sp.save_npz(out/f'X_{name}.npz',matrix)
for name,target in [('train',y_train),('validation',y_val),('test',y_test)]:
    np.save(out/f'y_{name}.npy',target)
joblib.dump(processor,out/'transform_pipeline.joblib')
feature_names=processor.get_feature_names_out()
pd.DataFrame({'feature':feature_names}).to_csv(out/'feature_names.csv',index=False,encoding='utf-8-sig')
print('Đã lưu ma trận, nhãn, mã lượt khám và pipeline vào:',out)
print('Số chiều sau mã hóa:',len(feature_names))

## 5. Kiểm tra biến ít biến thiên
Phân tích số giá trị khác nhau **chỉ trên train**. Chưa loại tự động; kết quả là đề xuất để đối chiếu chất lượng mô hình về sau.

In [ ]:
unique_train=X_train.nunique(dropna=False).sort_values()
low_var=pd.DataFrame({'cot':unique_train.index,'so_gia_tri_phan_biet_train':unique_train.values})
low_var['de_xuat']=np.where(low_var.so_gia_tri_phan_biet_train<=1,'Cân nhắc loại (hằng số)','Giữ để xem xét')
display(low_var.head(20))
low_var.to_csv(report/'feature_cardinality_train.csv',index=False,encoding='utf-8-sig')

## 6. PCA thăm dò trên **biến số** (không áp dụng mặc định cho mô hình)
Với One-Hot, ma trận thường rất thưa; PCA tiêu chuẩn trên toàn bộ One-Hot sẽ làm dày và tốn RAM. Vì vậy bước này chỉ thử PCA trên khối numeric đã chuẩn hóa, fit train, để xem số thành phần giải thích phương sai. Không dùng thông tin validation/test khi fit.

Sản phẩm chính cho mô hình vẫn là ma trận One-Hot + numeric được lưu ở mục 4.

In [ ]:
numeric_transformer=processor.named_transformers_['numeric']
Ztr=numeric_transformer.transform(X_train[num_cols])
component_count=min(len(num_cols),Ztr.shape[0])
pca=PCA(n_components=component_count,random_state=RANDOM_STATE)
pca.fit(Ztr)
pca_report=pd.DataFrame({'thanh_phan':np.arange(1,component_count+1),
    'phuong_sai_giai_thich':pca.explained_variance_ratio_,
    'tich_luy':np.cumsum(pca.explained_variance_ratio_)})
display(pca_report)
pca_report.to_csv(report/'pca_numeric_explained_variance.csv',index=False,encoding='utf-8-sig')
joblib.dump(pca,report/'pca_numeric_exploratory.joblib')
need_90=int(np.searchsorted(pca_report.tich_luy.to_numpy(),0.90)+1)
print('Số thành phần numeric để đạt >=90% phương sai:',need_90,'/',len(num_cols))

## 7. Decision log, đánh giá tính toàn vẹn và kết luận
Decision log lưu cả **lý do** và **hệ quả** của mỗi lựa chọn. Bước 7 của đồ án sẽ so sánh mô hình với baseline và các lựa chọn tiền xử lý, không kết luận PCA cải thiện kết quả khi chưa kiểm nghiệm.

In [ ]:
decision_log=pd.DataFrame([
 ['D04-01','Chia theo patient_nbr, 70/15/15 xấp xỉ','Một bệnh nhân có nhiều lượt khám; tránh rò rỉ giữa tập','Số lượt/nhãn từng tập, không trùng patient_nbr'],
 ['D04-02','Bỏ ID, nhãn, raw readmitted và mã discharge khỏi đầu vào','Ngăn nhãn/định danh lọt vào X; thận trọng với trạng thái xuất viện','Số biến đầu vào, giảm rủi ro leakage'],
 ['D04-03','Numeric median + StandardScaler fit train','Không học tham số từ validation/test','Giá trị đã chuẩn hóa và khả năng tái lập'],
 ['D04-04','Danh mục Unknown + OneHot min_frequency=50 fit train','Xử lý loại hiếm, danh mục chưa thấy, tránh bùng số chiều','Số cột OneHot và nhóm hiếm'],
 ['D04-05','Lưu ma trận sparse + pipeline','Giữ RAM trong giới hạn và tái lập biến đổi','Dung lượng, trật tự cột, ma trận phân lớp'],
 ['D04-06','PCA chỉ thăm dò trên numeric train','PCA toàn bộ OneHot dễ tốn RAM; giảm chiều cần đánh giá sau','Phương sai giải thích, chưa chứng minh hiệu quả dự đoán'],
],columns=['ma','quyet_dinh','ly_do','anh_huong'])
decision_log.to_csv(report/'decision_log_step4.csv',index=False,encoding='utf-8-sig')
checks={'train_val_patient_disjoint':patient_sets['train'].isdisjoint(patient_sets['validation']),
        'train_test_patient_disjoint':patient_sets['train'].isdisjoint(patient_sets['test']),
        'val_test_patient_disjoint':patient_sets['validation'].isdisjoint(patient_sets['test']),
        'all_encounters_accounted_for':sum(len(x) for x in splits.values())==len(df),
        'same_feature_dim':Xtr.shape[1]==Xva.shape[1]==Xte.shape[1],
        'all_labels_binary':all(set(np.unique(y)).issubset({0,1}) for y in [y_train,y_val,y_test])}
assert all(checks.values()),checks
with open(report/'integrity_checks.json','w',encoding='utf-8') as f:
    json.dump(checks,f,ensure_ascii=False,indent=2)
print('Đã lưu:',report/'decision_log_step4.csv')
print('Đã lưu:',report/'split_summary.csv')
print('Đã lưu:',out/'X_train.npz')
print('Kích thước train:',Xtr.shape,'| validation:',Xva.shape,'| test:',Xte.shape)
print('Kiểm tra toàn vẹn:',checks)
print('HOÀN THÀNH BƯỚC 4.')

### Nộp/giữ lại
- `notebooks/04_transformation.ipynb` (chạy xong và lưu)
- `data/processed/step4/`: `.npz`, nhãn `.npy`, ID/nhãn từng tập, pipeline và tên đặc trưng
- `reports/transformation/`: bảng chia tập, kiểm tra biến, PCA, nhật ký quyết định, kiểm tra tính toàn vẹn

**Lưu ý với báo cáo:** tách theo bệnh nhân tránh lẫn cùng bệnh nhân giữa các tập, nhưng dữ liệu không có mốc thời gian chính xác; không khẳng định đã kiểm soát được mọi dạng rò rỉ theo thời gian. Bước kế tiếp trong lộ trình môn học là xây kho dữ liệu PostgreSQL (star schema/OLAP).

# Bước 4 — Transformation & giảm chiều (Đồ án 5)
**Dự đoán tái nhập viện trong 30 ngày** · Chạy hoàn toàn trên máy cá nhân.

**Nguyên tắc:** Chia **theo bệnh nhân trước**; chỉ học imputer/scaler/encoder/PCA trên **train**. Không dùng `readmitted`, `readmitted_30d`, `patient_nbr`, `encounter_id` làm đặc trưng. Cần kiểm tra tính sẵn có của biến tại thời điểm **xuất viện** trước khi dùng trong mô hình.

Notebook này tạo ba tập train/validation/test, xử lý biến số và danh mục, lưu ma trận thưa và ghi nhật ký quyết định. Không huấn luyện mô hình phân lớp tại bước này.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import scipy.sparse as sp
from IPython.display import display
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
import joblib

RANDOM_STATE = 42
cwd = Path.cwd()
root = next((p for p in [cwd, *cwd.parents] if (p/'data'/'processed'/'step3_cleaned.csv').exists()), None)
if root is None:
    raise FileNotFoundError('Không tìm thấy data/processed/step3_cleaned.csv. Hãy mở JupyterLab từ D:/Readmission_Project.')
source = root/'data'/'processed'/'step3_cleaned.csv'
out = root/'data'/'processed'/'step4'
report = root/'reports'/'transformation'
out.mkdir(parents=True, exist_ok=True)
report.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(source, low_memory=False)
assert {'encounter_id','patient_nbr','readmitted','readmitted_30d'}.issubset(df.columns)
assert df['encounter_id'].is_unique, 'Encounter ID cần được kiểm tra trước khi chạy.'
assert df['readmitted_30d'].isin([0,1]).all()
print('Nguồn:',source)
print('Kích thước:',df.shape)
print('Bệnh nhân:',df.patient_nbr.nunique())

## 1. Chọn đặc trưng theo thời điểm dự đoán
Bài toán giả định dự đoán **ngay lúc xuất viện**. Loại biến định danh, nhãn, và mã tình trạng xuất viện (`discharge_disposition_id`) trong bản mô hình cơ sở để tránh dùng thông tin kết cục/chăm sóc sau xuất viện sai mục đích. Các biến được chọn phải được xác nhận có sẵn vào thời điểm dự đoán; nếu triển khai sớm hơn xuất viện, phải xét lại biến thời gian nằm viện và điều trị.

**Chú ý:** Đây là danh sách đặc trưng để thử nghiệm, không khẳng định chúng có tính nhân quả.

In [ ]:
numeric_candidates = [
    'time_in_hospital','num_lab_procedures','num_procedures','num_medications',
    'number_outpatient','number_emergency','number_inpatient','number_diagnoses',
    'A1Cresult_tested','max_glu_serum_tested'
]
category_candidates = [
    'race','gender','age','admission_type_id_category','admission_source_id_category',
    'payer_code','medical_specialty','diag_1','diag_2','diag_3',
    'max_glu_serum','A1Cresult','change','diabetesMed',
    'insulin','metformin','repaglinide','nateglinide','chlorpropamide','glimepiride',
    'acetohexamide','glipizide','glyburide','tolbutamide','pioglitazone',
    'rosiglitazone','acarbose','miglitol','troglitazone','tolazamide',
    'examide','citoglipton','glyburide-metformin','glipizide-metformin',
    'glimepiride-pioglitazone','metformin-rosiglitazone','metformin-pioglitazone'
]
num_cols=[c for c in numeric_candidates if c in df.columns]
cat_cols=[c for c in category_candidates if c in df.columns]
missing=[c for c in numeric_candidates+category_candidates if c not in df.columns]
if missing: print('Lưu ý: Một số cột ứng viên không tồn tại:',missing)
assert len(num_cols)>0 and len(cat_cols)>0
assert not set(['encounter_id','patient_nbr','readmitted','readmitted_30d','discharge_disposition_id']) & set(num_cols+cat_cols)
features=num_cols+cat_cols
print('Số biến số:',len(num_cols),'| biến phân loại:',len(cat_cols),'| tổng:',len(features))
print('Các biến đầu vào:',features)

## 2. Chia dữ liệu theo bệnh nhân (70% / 15% / 15% lượt khám, xấp xỉ)
Dùng GroupShuffleSplit hai lần: nhóm bệnh nhân không được xuất hiện ở nhiều tập. Tỉ lệ thực tế tính theo lượt khám có thể chênh nhẹ; notebook sẽ báo cáo tỷ lệ và mức cân bằng nhãn.

In [ ]:
groups=df['patient_nbr']
idx=np.arange(len(df))
split1=GroupShuffleSplit(n_splits=1,test_size=0.30,random_state=RANDOM_STATE)
train_idx,temp_idx=next(split1.split(idx,df['readmitted_30d'],groups))
split2=GroupShuffleSplit(n_splits=1,test_size=0.50,random_state=RANDOM_STATE)
val_rel,test_rel=next(split2.split(temp_idx,df.iloc[temp_idx]['readmitted_30d'],groups.iloc[temp_idx]))
val_idx=temp_idx[val_rel]
test_idx=temp_idx[test_rel]
splits={'train':train_idx,'validation':val_idx,'test':test_idx}
patient_sets={k:set(df.iloc[ix]['patient_nbr']) for k,ix in splits.items()}
assert all(patient_sets[a].isdisjoint(patient_sets[b]) for a,b in [('train','validation'),('train','test'),('validation','test')])
assert len(set(np.concatenate(list(splits.values()))))==len(df)
rows=[]
for name,ix in splits.items():
    subset=df.iloc[ix]
    rows.append({'tap':name,'so_luot':len(ix),'ty_le_luot_%':round(100*len(ix)/len(df),2),
                 'so_benh_nhan':subset.patient_nbr.nunique(),
                 'nhan_1':int(subset.readmitted_30d.sum()),
                 'ty_le_nhan_1_%':round(100*subset.readmitted_30d.mean(),3)})
summary=pd.DataFrame(rows)
display(summary)
summary.to_csv(report/'split_summary.csv',index=False,encoding='utf-8-sig')
for name,ix in splits.items():
    df.iloc[ix][['encounter_id','patient_nbr','readmitted_30d']].to_csv(out/f'{name}_ids_labels.csv',index=False)
print('Xác nhận: KHÔNG bệnh nhân nào giao nhau giữa train/validation/test.')

## 3. Biến đổi chỉ fit trên train
- Numeric: median imputation → StandardScaler.
- Category: thay thiếu bằng `Unknown` → OneHotEncoder(`handle_unknown='ignore'`) để không lỗi khi tập val/test có danh mục mới. Giới hạn tần suất hiếm (`min_frequency=50`) để giảm số cột quá lớn.
- Chỉ `fit` trên train, sau đó `transform` validation/test.

**Cảnh báo:** Không gộp train+validation+test trước khi tính median, trung bình/độ lệch chuẩn hay danh mục OneHot.

In [ ]:
X_train=df.iloc[train_idx][features].copy()
X_val=df.iloc[val_idx][features].copy()
X_test=df.iloc[test_idx][features].copy()
y_train=df.iloc[train_idx]['readmitted_30d'].astype('int8').to_numpy()
y_val=df.iloc[val_idx]['readmitted_30d'].astype('int8').to_numpy()
y_test=df.iloc[test_idx]['readmitted_30d'].astype('int8').to_numpy()
for frame in [X_train,X_val,X_test]:
    for c in num_cols: frame[c]=pd.to_numeric(frame[c],errors='coerce')
    for c in cat_cols: frame[c]=frame[c].astype('string').fillna('Unknown').replace({'?':'Unknown','':'Unknown'}).astype(str)
num_pipe=Pipeline([('imputer',SimpleImputer(strategy='median')),('scaler',StandardScaler())])
cat_pipe=Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),
                   ('onehot',OneHotEncoder(handle_unknown='ignore',min_frequency=50,sparse_output=True))])
processor=ColumnTransformer([('numeric',num_pipe,num_cols),('categorical',cat_pipe,cat_cols)],
                            sparse_threshold=1.0)
Xtr=processor.fit_transform(X_train)
Xva=processor.transform(X_val)
Xte=processor.transform(X_test)
Xtr=sp.csr_matrix(Xtr); Xva=sp.csr_matrix(Xva); Xte=sp.csr_matrix(Xte)
assert Xtr.shape[1]==Xva.shape[1]==Xte.shape[1]
assert np.isfinite(Xtr.data).all() and np.isfinite(Xva.data).all() and np.isfinite(Xte.data).all()
print('Train:',Xtr.shape,'Validation:',Xva.shape,'Test:',Xte.shape)
print('Sparsity train:',round(100*(1-Xtr.nnz/(Xtr.shape[0]*Xtr.shape[1])),2),'%')

## 4. Xuất ma trận và pipeline
Ma trận đã biến đổi lưu dạng `.npz` (sparse) thay vì CSV khổng lồ; danh sách đặc trưng và pipeline `.joblib` cho phép biến đổi dữ liệu mới bằng cùng tham số. File `test` dùng để đánh giá cuối cùng, không dùng để chỉnh mô hình.

In [ ]:
for name,matrix in [('train',Xtr),('validation',Xva),('test',Xte)]:
    sp.save_npz(out/f'X_{name}.npz',matrix)
for name,target in [('train',y_train),('validation',y_val),('test',y_test)]:
    np.save(out/f'y_{name}.npy',target)
joblib.dump(processor,out/'transform_pipeline.joblib')
feature_names=processor.get_feature_names_out()
pd.DataFrame({'feature':feature_names}).to_csv(out/'feature_names.csv',index=False,encoding='utf-8-sig')
print('Đã lưu ma trận, nhãn, mã lượt khám và pipeline vào:',out)
print('Số chiều sau mã hóa:',len(feature_names))

## 5. Kiểm tra biến ít biến thiên
Phân tích số giá trị khác nhau **chỉ trên train**. Chưa loại tự động; kết quả là đề xuất để đối chiếu chất lượng mô hình về sau.

In [ ]:
unique_train=X_train.nunique(dropna=False).sort_values()
low_var=pd.DataFrame({'cot':unique_train.index,'so_gia_tri_phan_biet_train':unique_train.values})
low_var['de_xuat']=np.where(low_var.so_gia_tri_phan_biet_train<=1,'Cân nhắc loại (hằng số)','Giữ để xem xét')
display(low_var.head(20))
low_var.to_csv(report/'feature_cardinality_train.csv',index=False,encoding='utf-8-sig')

## 6. PCA thăm dò trên **biến số** (không áp dụng mặc định cho mô hình)
Với One-Hot, ma trận thường rất thưa; PCA tiêu chuẩn trên toàn bộ One-Hot sẽ làm dày và tốn RAM. Vì vậy bước này chỉ thử PCA trên khối numeric đã chuẩn hóa, fit train, để xem số thành phần giải thích phương sai. Không dùng thông tin validation/test khi fit.

Sản phẩm chính cho mô hình vẫn là ma trận One-Hot + numeric được lưu ở mục 4.

In [ ]:
numeric_transformer=processor.named_transformers_['numeric']
Ztr=numeric_transformer.transform(X_train[num_cols])
component_count=min(len(num_cols),Ztr.shape[0])
pca=PCA(n_components=component_count,random_state=RANDOM_STATE)
pca.fit(Ztr)
pca_report=pd.DataFrame({'thanh_phan':np.arange(1,component_count+1),
    'phuong_sai_giai_thich':pca.explained_variance_ratio_,
    'tich_luy':np.cumsum(pca.explained_variance_ratio_)})
display(pca_report)
pca_report.to_csv(report/'pca_numeric_explained_variance.csv',index=False,encoding='utf-8-sig')
joblib.dump(pca,report/'pca_numeric_exploratory.joblib')
need_90=int(np.searchsorted(pca_report.tich_luy.to_numpy(),0.90)+1)
print('Số thành phần numeric để đạt >=90% phương sai:',need_90,'/',len(num_cols))

## 7. Decision log, đánh giá tính toàn vẹn và kết luận
Decision log lưu cả **lý do** và **hệ quả** của mỗi lựa chọn. Bước 7 của đồ án sẽ so sánh mô hình với baseline và các lựa chọn tiền xử lý, không kết luận PCA cải thiện kết quả khi chưa kiểm nghiệm.

In [ ]:
decision_log=pd.DataFrame([
 ['D04-01','Chia theo patient_nbr, 70/15/15 xấp xỉ','Một bệnh nhân có nhiều lượt khám; tránh rò rỉ giữa tập','Số lượt/nhãn từng tập, không trùng patient_nbr'],
 ['D04-02','Bỏ ID, nhãn, raw readmitted và mã discharge khỏi đầu vào','Ngăn nhãn/định danh lọt vào X; thận trọng với trạng thái xuất viện','Số biến đầu vào, giảm rủi ro leakage'],
 ['D04-03','Numeric median + StandardScaler fit train','Không học tham số từ validation/test','Giá trị đã chuẩn hóa và khả năng tái lập'],
 ['D04-04','Danh mục Unknown + OneHot min_frequency=50 fit train','Xử lý loại hiếm, danh mục chưa thấy, tránh bùng số chiều','Số cột OneHot và nhóm hiếm'],
 ['D04-05','Lưu ma trận sparse + pipeline','Giữ RAM trong giới hạn và tái lập biến đổi','Dung lượng, trật tự cột, ma trận phân lớp'],
 ['D04-06','PCA chỉ thăm dò trên numeric train','PCA toàn bộ OneHot dễ tốn RAM; giảm chiều cần đánh giá sau','Phương sai giải thích, chưa chứng minh hiệu quả dự đoán'],
],columns=['ma','quyet_dinh','ly_do','anh_huong'])
decision_log.to_csv(report/'decision_log_step4.csv',index=False,encoding='utf-8-sig')
checks={'train_val_patient_disjoint':patient_sets['train'].isdisjoint(patient_sets['validation']),
        'train_test_patient_disjoint':patient_sets['train'].isdisjoint(patient_sets['test']),
        'val_test_patient_disjoint':patient_sets['validation'].isdisjoint(patient_sets['test']),
        'all_encounters_accounted_for':sum(len(x) for x in splits.values())==len(df),
        'same_feature_dim':Xtr.shape[1]==Xva.shape[1]==Xte.shape[1],
        'all_labels_binary':all(set(np.unique(y)).issubset({0,1}) for y in [y_train,y_val,y_test])}
assert all(checks.values()),checks
with open(report/'integrity_checks.json','w',encoding='utf-8') as f:
    json.dump(checks,f,ensure_ascii=False,indent=2)
print('Đã lưu:',report/'decision_log_step4.csv')
print('Đã lưu:',report/'split_summary.csv')
print('Đã lưu:',out/'X_train.npz')
print('Kích thước train:',Xtr.shape,'| validation:',Xva.shape,'| test:',Xte.shape)
print('Kiểm tra toàn vẹn:',checks)
print('HOÀN THÀNH BƯỚC 4.')

### Nộp/giữ lại
- `notebooks/04_transformation.ipynb` (chạy xong và lưu)
- `data/processed/step4/`: `.npz`, nhãn `.npy`, ID/nhãn từng tập, pipeline và tên đặc trưng
- `reports/transformation/`: bảng chia tập, kiểm tra biến, PCA, nhật ký quyết định, kiểm tra tính toàn vẹn

**Lưu ý với báo cáo:** tách theo bệnh nhân tránh lẫn cùng bệnh nhân giữa các tập, nhưng dữ liệu không có mốc thời gian chính xác; không khẳng định đã kiểm soát được mọi dạng rò rỉ theo thời gian. Bước kế tiếp trong lộ trình môn học là xây kho dữ liệu PostgreSQL (star schema/OLAP).